In [9]:
import numpy as np
from astropy import units as u
from astropy import time

import numpy as np

from poliastro import iod
from poliastro.bodies import Body,Mars, Earth, Venus, Jupiter, Saturn, Uranus, Neptune, Sun, Europa, Ganymede, Callisto, Io, Titan
from poliastro.ephem import Ephem
from poliastro.maneuver import Maneuver
from poliastro.twobody import Orbit
from poliastro.util import time_range
from poliastro.plotting import OrbitPlotter3D, StaticOrbitPlotter
import math
import matplotlib.pyplot as plt
# More info: https://plotly.com/python/renderers/
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

from astropy.coordinates import solar_system_ephemeris


solar_system_ephemeris.set("jpl")


<ScienceState solar_system_ephemeris: 'jpl'>

In [ ]:
def parse_spenvis_spe(path):
    energies = None
    rows = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            # ENERGY line: 'ENERGY', 27, 0.06, 0.10, ... , 'MeV'
            if line.startswith("'ENERGY'"):
                parts = [p.strip() for p in line.split(",")]
                # parts[0]="'ENERGY'", parts[1]=27, parts[2:2+27]=energies, last="'MeV'"
                nE = int(parts[1])
                energies = np.array([float(x) for x in parts[2:2+nE]], dtype=float)

            # data rows (in your file they start with -1.0000E+00, -1.0000E+00, then 27 flux values)
            if line.startswith("-1.0000E+00, -1.0000E+00"):
                vals = [float(x) for x in line.split(",")]
                rows.append(vals)

    if energies is None:
        raise ValueError("Didn't find an ENERGY line in the file.")

    data = np.array(rows, dtype=float)
    # data columns: [B, L, flux(E1), flux(E2), ...]
    flux = data[:, 2:]  # shape (N, 27)
    return energies, flux

def kepler_r_vs_time(apogee_km, perigee_km, period_hours, dt_s, N):
    # Orbit elements from apo/peri
    ra = apogee_km
    rp = perigee_km
    a = 0.5 * (ra + rp)
    e = (ra - rp) / (ra + rp)

    T = period_hours * 3600.0
    n = 2.0 * np.pi / T

    t = np.arange(N, dtype=float) * dt_s
    M = (n * t) % (2.0 * np.pi)  # start at true anomaly 0 => perijove, so M0=0

    # Solve Kepler: M = E - e sin E (Newton)
    E = M.copy()
    for _ in range(30):
        f = E - e * np.sin(E) - M
        fp = 1.0 - e * np.cos(E)
        dE = -f / fp
        E += dE
        if np.max(np.abs(dE)) < 1e-12:
            break

    r = a * (1.0 - e * np.cos(E))  # km
    return r

def flux_vs_radius_from_spe(path_spe, dt_s=360.0, E0_MeV=2.0, RJ_km=71492.0):
    energies, flux = parse_spenvis_spe(path_spe)

    # pull orbit params from your header (hardcode from file, or parse similarly)
    apogee_km = 6.791740e6
    perigee_km = 1.072380e5
    period_hours = 1.023800e3

    r_km = kepler_r_vs_time(apogee_km, perigee_km, period_hours, dt_s, N=flux.shape[0])
    r_RJ = r_km / RJ_km

    # choose the column corresponding to E0 (Integral Flux > E0)
    idx = np.argmin(np.abs(energies - E0_MeV))
    if abs(energies[idx] - E0_MeV) > 1e-6:
        raise ValueError(f"E0={E0_MeV} MeV not found; nearest is {energies[idx]} MeV")

    F = flux[:, idx]  # integral flux > E0
    return r_RJ, F, energies



In [ ]:
r_RJ, F_gt2, energies = flux_vs_radius_from_spe("spenvis_files/spenvis_spe.txt", dt_s=360.0, E0_MeV=2.0)
